polar-triplet world map (elevation lapse rates + per-range triplets)

Builds every component of the composite from the pipeline products and assembles the page with
`gsro_analysis.world_maps.plot_lapse_rate_triplet_map`. Run top to bottom for a new dataset version;
when the sweep and legend already exist, the setup cell and the composite cell [3] are enough.

```
pipeline products (not built here)         this notebook                                              outputs (figures/<version>/)
──────────────────────────────────         ─────────────                                              ────────────────────────────
mountain-range cube ──────────────┬──► [1] per-range triplet sweep ────────────────────────────────► triplets/<stem>.png  (one per range)
  (0_aggregate_by_mountain_range)  └──► [2] triplet legend ────────────────────────────────────────► polar_triplet_legend.png
range metrics table ─────────────────► choropleth: snowmelt_lapse_rate_per_100m where n ≥ 10 ──────┐
  (0_aggregate_by_mountain_range)                                                                    │
label_layout.csv (curated) ──────────► which 40 ranges get a triplet + anchors (Robinson metres) ───┼──► [3] world_maps.plot_lapse_rate_triplet_map
data/global_hillshade_robinson.tif ──► base map (grey 1–231) + generated 60° × 30° graticule ───────┤        └──► global_lapse_rates_with_triplets_map.png
gsro_analysis.colorbars.elevation_delay ► colorbar (YlGnBu, 0–8 days per 100 m) ────────────────────┘
```

Everything is versioned by dataset version (`config.version` from `settings.load_config()`): inputs are read
from `data/aggregation/<version>/` and `results/<version>/`, outputs go to `analyses/mountain_ranges/figures/<version>/`.

In [ ]:
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

from gsro_analysis import aggregate, paths, plotting, settings, world_maps
from gsro_analysis.plotting import label_angle, major_tick_radii, rgrid_labels, rgrid_labels_blank, rgrid_vals

config = settings.load_config()          # the dataset version lives in settings.CONFIG_FILE
version = config.version
figdir = paths.figdir('mountain_ranges', version)

# the mountain-range cube written by 0_aggregate_by_mountain_range.ipynb (range x elevation x aspect x chili_class x water_year)
mountain_ranges_cube_ds = xr.open_dataset(paths.aggregation_dir('mountain_ranges', version) / 'all_mountain_ranges_fcf_lte_50.nc')
mountain_ranges_cube_ds

In [ ]:
# --- the analyses' view of the cube, the same rules in every mountain-range notebook -----------------------------
MIN_PIXELS_PER_BIN = 100          # a (range, elevation, aspect) bin with this many pixels or fewer is masked
MIN_YEAR_FRACTION_OF_BIN = 0.3    # a bin-year keeps its value only with more than 30 % of the bin's median pixels
MIN_YEAR_FRACTION_OF_RANGE = 0.1  # a range-year mean anomaly needs at least 10 % of the range's median pixels

mountain_ranges_ds = aggregate.collapse(mountain_ranges_cube_ds)              # CHILI classes folded (count-weighted, exact)
mountain_ranges_ds = aggregate.threshold(mountain_ranges_ds, MIN_PIXELS_PER_BIN + 1)
enough_pixels_this_year = mountain_ranges_ds['runoff_onset_n'] > MIN_YEAR_FRACTION_OF_BIN * mountain_ranges_ds['runoff_onset_median_n']
for var in ['runoff_onset', 'runoff_onset_anomaly', 'runoff_onset_std', 'runoff_onset_anomaly_std']:
    mountain_ranges_ds[var] = mountain_ranges_ds[var].where(enough_pixels_this_year)

# tropical-Andes rule: in South American ranges north of 20 S, median onsets of 250 DOWY or later below 5000 m are
# late-season artefacts
tropical_andes = (mountain_ranges_ds['continent'] == 'South America') & (mountain_ranges_ds['centroid_latitude'] > -20)
keep = (mountain_ranges_ds['runoff_onset_median'] < 250) | (mountain_ranges_ds['elevation'] > 5000) | ~tropical_andes
for var in ['runoff_onset_median', 'runoff_onset_median_std', 'runoff_onset_mad', 'runoff_onset_mad_std',
            'runoff_onset', 'runoff_onset_anomaly', 'runoff_onset_std', 'runoff_onset_anomaly_std']:
    mountain_ranges_ds[var] = mountain_ranges_ds[var].where(keep)

# each aspect's deviation from the per-elevation median across aspects (the triplets' third panel), masked where an
# aspect is missing
elev_relative_da = aggregate.elevation_relative(mountain_ranges_ds['runoff_onset_median'])
mountain_ranges_ds['runoff_onset_elev_relative'] = elev_relative_da.where(~elev_relative_da.isnull().any('aspect'))

# the pixel-weighted mean onset anomaly per range and water year; a range-year with fewer valid pixels than
# MIN_YEAR_FRACTION_OF_RANGE of the range's median pixels is masked (the same rule as the metrics table)
range_mean_anomaly_da = aggregate.weighted_mean(mountain_ranges_ds, 'runoff_onset_anomaly', ['elevation', 'aspect'])
valid_fraction_da = mountain_ranges_ds['runoff_onset_n'].sum(['elevation', 'aspect']) / mountain_ranges_ds['runoff_onset_median_n'].sum(['elevation', 'aspect'])
mountain_ranges_ds['runoff_onset_mean_anomaly'] = range_mean_anomaly_da.where(valid_fraction_da >= MIN_YEAR_FRACTION_OF_RANGE)

# ranges without any data left are dropped
has_data = mountain_ranges_ds['runoff_onset_median'].notnull().any(('elevation', 'aspect'))
mountain_ranges_ds = mountain_ranges_ds.sel(mountain_range=has_data)
mountain_ranges_ds

In [ ]:
# polar axes want radians; the cube stores aspect in degrees
mountain_ranges_ds = mountain_ranges_ds.assign_coords(aspect=np.deg2rad(mountain_ranges_ds['aspect']))
mountain_ranges_ds

In [ ]:
n_years = len(mountain_ranges_ds['water_year'])
# the per-range metrics table (0_aggregate_by_mountain_range.ipynb) joined to the GMBA polygons: the map's choropleth
metrics_df = pd.read_csv(paths.resultsdir('mountain_ranges', version) / 'mountain_range_metrics.csv')
gmba_gdf = gpd.read_file('zip+' + settings.GMBA_URL)      # GMBA Inventory v2.0 standard 300, read straight from EarthEnv
gmba_stats_gdf = gmba_gdf[['GMBA_V2_ID', 'MapName', 'Level_04', 'Hier_Lvl', 'Area', 'Perimeter', 'geometry']].merge(
    metrics_df.drop(columns=['name']), on='GMBA_V2_ID', how='inner')
print(f'{version}: {mountain_ranges_ds.sizes["mountain_range"]} ranges, {n_years} water years; {len(gmba_stats_gdf)} GMBA rows with metrics')
gmba_stats_gdf

## [1] Per-range triplet sweep

One PNG per range (`triplets/<stem>.png`, gitignored, regenerable): three polar panels on a transparent
figure background — median runoff onset (viridis, 100–300 DOWY), median absolute deviation (Reds, 0–30 days)
and elevation-normalized timing (RdBu, ±15 days) — with aspect around the circle (N up, clockwise) and
elevation along the radius (0 m at the rim, 7000 m at the centre; black rings every 1000 m). Also
the raw material of the all-ranges overview in `topography.ipynb`. `REBUILD_SWEEP = False` skips the ~10 min sweep when every range already has
a PNG in this version's folder.

In [ ]:
# Create directory if it doesn't exist
triplet_png_dir = paths.figdir('mountain_ranges', config.version, 'triplets')


# Define metrics
metrics = [
    ('runoff_onset_median', 'viridis', 100, 300),
    ('runoff_onset_mad', 'Reds', 0, 30),
    ('runoff_onset_elev_relative', 'RdBu', -15, 15)
]

cbar_labels = ['10-year median runoff onset [DOWY]','10-year MAD of runoff onset [days]','Runoff onset difference from elevation median [days]']

# Get all mountain ranges to process
all_ranges = mountain_ranges_ds.mountain_range.values

REBUILD_SWEEP = False   # False: skip when every range already has a PNG here (the composite only needs the files)
if not REBUILD_SWEEP and len(list(triplet_png_dir.glob('*.png'))) >= len(all_ranges):
    print(f'{len(all_ranges)} triplet PNGs already in {triplet_png_dir}; skipping the sweep (REBUILD_SWEEP = False)')
    all_ranges = []

# Process each mountain range individually
for mountain in all_ranges:
    
    # if mountain != 'Tian Shan':
    #     continue
        
    # Create figure with transparent background
    fig, axs = plt.subplots(1, 3, figsize=(6, 2), 
                           subplot_kw={'projection': 'polar'},
                           dpi=300,
                           )
    
    # Make figure background transparent but keep subplot backgrounds
    fig.patch.set_alpha(0.0)  # Transparent figure background
    
    # Get data for this mountain range
    data = mountain_ranges_ds.sel(mountain_range=mountain)
    
    r_min = 0
    r_max = 7000
    
    # Plot each metric
    for metric_idx, (metric, cmap, vmin, vmax) in enumerate(metrics):
        ax = axs[metric_idx]
        
        # Plot data
        im = data[metric].plot.pcolormesh(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, 
                              add_colorbar=False, yincrease=False, edgecolors='face')
        
        # Keep darkgrey background inside circle - this stays solid
        ax.set_facecolor('darkgrey')
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)

        # Add major tick radius circles
        for major_tick_radius in major_tick_radii:
            ax.plot(np.linspace(0, 2*np.pi, 100), np.ones(100)*major_tick_radius, 
                   color='black', linestyle='-', alpha=1, linewidth=1)
        
        # Remove labels and ticks for clean appearance
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title('')
        ax.set_thetagrids([0,45,90,135,180,225,270,315,360],
                         labels=['','','','','','','','',''])
        ax.set_rgrids(rgrid_vals, labels=rgrid_labels_blank, angle=label_angle,
                     fontsize=7, ha='left', va='bottom', zorder=0)
        
        # Add grid lines
        ax.xaxis.grid(True, which="major", linestyle=":", color='gray', alpha=0.8, linewidth=1)
        [x.set_linewidth(1.2) for x in ax.spines.values()]
        ax.set_thetamin(0)
        ax.set_thetamax(360)
        ax.set_rlim(bottom=r_max, top=r_min)
            
    # Adjust layout
    #plt.tight_layout()
    fig.subplots_adjust(wspace=0.05)
    
    # Create safe filename (replace problematic characters)
    safe_filename = mountain.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_').replace('-', '_').replace("*","_")
    safe_filename = safe_filename
    
    # Saved with a transparent figure background but solid circles, for embedding on maps.
    fig.savefig(triplet_png_dir / f'{safe_filename}.png', 
                dpi=300, 
                bbox_inches='tight', 
                transparent=False,  # Keep this False
                facecolor='none')   # This makes the figure background transparent


    # Close figure to free memory
    plt.close(fig)
    
    print(f"Saved: {safe_filename}")

print(f"Completed processing {len(all_ranges)} mountain ranges")

## [2] Triplet legend

The three annotated panels (one representative range, aspect labels, elevation ticks, one colorbar each)
that sit in the composite's lower-right corner as `polar_triplet_legend.png`. Titles carry the water-year
count of the cube.

In [ ]:
def create_simple_polar_legend():
    """Create a simple 3-panel legend using real data averaged across all mountain ranges"""
    
    fig, axs = plt.subplots(1, 3, figsize=(12, 5), 
                           subplot_kw={'projection': 'polar'}, 
                           dpi=300)
    
    # Use real data - take mean across all mountain ranges
    legend_range = 'Tian Shan' if 'Tian Shan' in mountain_ranges_ds.mountain_range.values else str(mountain_ranges_ds.mountain_range.values[0])
    global_subset_mean_data = mountain_ranges_ds.sel(mountain_range=[legend_range]).mean(dim='mountain_range')
    
    # Plot each metric using the same approach as your main code
    datasets = ['runoff_onset_median', 'runoff_onset_mad', 'runoff_onset_elev_relative']
    cmaps = ['viridis', 'Reds', 'RdBu']
    vmins = [100, 0, -15]
    vmaxs = [300, 30, 15]
    titles = [f'{n_years}-yr median runoff onset', f'{n_years}-yr median absolute deviation', 'Elevation-normalized timing']
    cbar_labels = ['day of water year', 'days', 'days']
    
    for i, (ax, metric, cmap, vmin, vmax, title, cbar_label) in enumerate(zip(axs, datasets, cmaps, vmins, vmaxs, titles, cbar_labels)):
        
        # Plot data using the same .plot() method as your main code
        im = global_subset_mean_data[metric].plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, 
                                          add_colorbar=False, yincrease=False)
        
        # Apply exact styling to match your triplet plots
        ax.set_facecolor('darkgrey')
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)

        # Add ONLY the major tick radius circles (solid black lines)
        for major_tick_radius in major_tick_radii:
            ax.plot(np.linspace(0, 2*np.pi, 100), np.ones(100)*major_tick_radius, 
                   color='black', linestyle='-', alpha=1, linewidth=1)

        ax.set_xlabel('')
        ax.set_ylabel('')
        
        # Add complete aspect labels
        ax.set_thetagrids([0,45,90,135,180,225,270,315,360],
                         labels=['N','NE','E','SE','S','SW','W','NW',''], fontsize=14)

        # Use the same rgrids setup as your main plots
        # ax.set_rgrids(rgrid_vals, labels=rgrid_labels, angle=180,
        #              fontsize=10, ha='left', va='bottom', zorder=0)
        
        ax.set_rgrids(rgrid_vals, labels=[], angle=180)

        
        horizontal_offset = 2500  # Adjust this value to control how far right the labels appear

        for radius, label in zip(rgrid_vals, rgrid_labels):
            # Convert elevation to plot radius (axis is inverted: center=8000m, edge=0m)
            
            if label == '':
                continue  # Skip empty labels
            
            if (label == '9000m') or (label == '8000m'):
                continue
            
            plot_radius = 7000 - radius
            
            # Bottom of circle is at (0, -plot_radius) in Cartesian space
            y_position = -plot_radius
            x_label = horizontal_offset
            
            # Add horizontal line from bottom of circle to label
            ax.plot([0, x_label], [y_position, y_position], 
                color='black', linestyle='-', linewidth=1, 
                transform=ax.transData._b, clip_on=False, zorder=4)
            
            # Add text label
            text = ax.text(x_label+200, y_position, label, 
                fontsize=12, ha='left', va='center',
                transform=ax.transData._b, clip_on=False, zorder=4)
            
            text.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])

        # Match your exact grid styling from the main plots
        ax.xaxis.grid(True, which="both", linestyle=":", color='gray', alpha=0.8, linewidth=1)
        
        # Keep spine styling consistent
        [x.set_linewidth(1.2) for x in ax.spines.values()]
        ax.set_thetamin(0)
        ax.set_thetamax(360)
        ax.set_rlim(bottom=7000, top=0)
        
        # Add title with larger font and reduced padding
        ax.set_title(title, fontsize=18, pad=15)
        
        # Add horizontal colorbar underneath each plot
        if metric == 'runoff_onset_mad':
            cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                            shrink=0.9, pad=0.1, aspect=10,extend='max')
        else:
            cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                            shrink=0.9, pad=0.1, aspect=10,extend='both') 
        cbar.set_label(cbar_label, fontsize=16)
        cbar.ax.tick_params(labelsize=14)

        # if elevation relative timing, set specific ticks to -15, 0, 15
        if metric == 'runoff_onset_elev_relative':
            cbar.set_ticks([-15, 0, 15])

        # Add text labels to colorbars
        if metric == 'runoff_onset_median':
            # Add month boundary lines
            month_boundaries = [124, 152, 183, 213, 244, 274]
            for boundary in month_boundaries:
                cbar.ax.axvline(x=boundary, ymin=0, ymax=1, color='white', 
                               linewidth=1.2, linestyle='--', zorder=10)
            
            # Month labels for DOWY colorbar
            month_labels = ['J', 'F', 'M', 'A', 'M', 'J', 'J']
            month_centers = [112, 138, 167.5, 198, 228.5, 259, 289.5]
            for center, label in zip(month_centers, month_labels):
                text = cbar.ax.text(center, 0.5, label, fontsize=16, ha='center', va='center',
                                color='white', weight='bold', transform=cbar.ax.transData)
                text.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='black')])

        elif metric == 'runoff_onset_mad':
            # Variability labels for MAD colorbar
            text1 = cbar.ax.text(1, 0.5, 'lower var.', fontsize=16, ha='left', va='center',
                                color='white', weight='bold', transform=cbar.ax.transData)
            text1.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='black')])
            text2 = cbar.ax.text(29, 0.5, 'higher var.', fontsize=16, ha='right', va='center',
                                color='white', weight='bold', transform=cbar.ax.transData)
            text2.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='black')])

        elif metric == 'runoff_onset_elev_relative':
            # Timing labels for elevation-normalized colorbar
            text1 = cbar.ax.text(-13, 0.5, 'earlier', fontsize=16, ha='left', va='center',
                                color='white', weight='bold', transform=cbar.ax.transData)
            text1.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='black')])
            text2 = cbar.ax.text(13, 0.5, 'later', fontsize=16, ha='right', va='center',
                                color='white', weight='bold', transform=cbar.ax.transData)
            text2.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground='black')])

    
    # Reduce overall whitespace
    plt.tight_layout(pad=0.5)
    return fig

# Create and save the legend using real data
legend_fig = create_simple_polar_legend()
#legend_fig.suptitle('Mountain range metrics by aspect and elevation', fontsize=20, y=1.1)
legend_fig.savefig(paths.figdir('mountain_ranges', config.version) / 'polar_triplet_legend.png', dpi=300, bbox_inches='tight')

## [3] The composite page: how it is built and every knob

**Page.** A 480 mm wide page in **page millimetres, origin top-left, y down** (`world_maps.Page`; the convention the published layout used, so every position below is a plain number in mm). The project page is 313.254 mm tall with the map item over the top 289.201 mm and a 24 mm white band below for the colorbar; the default 3 mm top margin extends it by ≈ 4 mm. The map item is drawn in World Robinson metres (`ESRI:54030`), extent `world_maps.PAGES['lapse_rates_with_triplets'].extent`, scale 1 : 70.9 M, i.e. `page.m_per_mm` ≈ 70.9 km per page millimetre. `world_maps.Page.to_page()` / `from_page()` convert between metres and page mm. With `top_margin_mm=3` (default, `world_maps.TOP_MARGIN_MM`) the page top is placed 3 mm above the highest label block and the pictures move with the map; `top_margin_mm=None` keeps the project page and instead shifts a block down when it would poke out.

**Layers, bottom to top.** hillshade (`data/global_hillshade_robinson.tif`, grey stretch 1–231, 0 = outside the ellipse = transparent, read decimated to 4 km pixels) → generated 60° × 30° graticule (white, 50 %, 0.45 pt dashed) → choropleth (`lapse_rate_fill`) → callouts → triplet images → names → legend picture (`polar_triplet_legend.png`, 1 mm frame) → frames. The colorbar is its own axes (`gsro_analysis.colorbars` preset, fonts scaled to the picture size).

**A label block** = the triplet image with its bottom-left corner 3 pt above the anchor, the range name one text line above the image (centred on it), and a straight 0.4 mm black callout from the nearest edge of the block to the polygon centroid. Anchors live in `analyses/mountain_ranges/label_layout.csv` in Robinson metres, columns `topo_x`, `topo_y` for this map (the other map has its own pair): **the anchor is the bottom-left corner of the block.** 41 ranges have `display_label = 1`; `show_topo = 0` hides a label on this map only (one here: Cordillera de la Costa); `display_map = 0` removes a range from the map entirely (four ranges). Insets are the triplet PNGs of [1], 53.8 mm wide (the project's `120·90e6/scale` pt rule; height follows the PNG); the filename stem is `world_maps.inset_stem(MapName, Level_04)`.

| I want to… | Do this |
| --- | --- |
| **move a label block** | edit `topo_x`, `topo_y` in `label_layout.csv`. 10 mm to the right = `+10 * page.m_per_mm`; the `placed` table below says where every block landed on the rendered page (`anchor_x_mm`, `anchor_y_mm`, `block_top`, `inset_*`), and `world_maps.PAGES['lapse_rates_with_triplets'].from_page(x_mm, y_mm)` converts a position on the *project* page to metres (on the resized page add the trim printed below to `y_mm`). Say why in the `note` column |
| **add a labelled range** | `display_label = 1`, `show_topo = 1` (and `show_anom` for the other map), both anchor pairs, make sure the inset PNG exists (built in the sweep above), re-run. A range missing from the CSV is drawn but never labelled |
| **hide a label on this map only** | `show_topo = 0` |
| **inset size** | `style=replace(STYLE_B, inset_width_mm=50)`; `None` = the project's rule |
| **name size / weight** | `size_pt` (12 — the fixed anchors allow no more before names touch the triplets above them in the stacked columns), `weight` ('bold'); colour `world_maps.TEXT_COLOR` (#323232), font `world_maps.FONT_FAMILIES` (Arial if installed, else Liberation Sans, else DejaVu Sans) |
| **buffer / halo / shadow** | `buffer_mm` 0.6 (`buffer_color` white), `halo_mm` 0 (a second, lighter stroke outside the buffer), `shadow` False (the project's blurred drop shadow, approximated) |
| **name-to-inset gap** | `gap_pt` 1.5 (project: 3) |
| **callouts** | `callout_lw_mm` 0.4, `callout_alpha` 1, `callout_origin` 'exterior' (nearest block edge) or 'centroid', `colored_callout` False |
| **the whole label style** | `style=` a `world_maps.LabelStyle`: `STYLE_B` (default), `STYLE_B_QGIS` (published), or `dataclasses.replace(world_maps.STYLE_B, size_pt=13)` |
| **the published look** | `style=world_maps.STYLE_B_QGIS, top_margin_mm=None` plus `pictures` with the colorbar back at x = 119.456 (it was moved 11.5 mm left so its label clears the legend) |
| **colorbar / legend position and frame** | `pictures=` a copy of `world_maps.PICTURES['lapse_rates_with_triplets']` with edited `rect=(x, y, w, h)` (project-page mm), `frame_mm`, `background`; the preset artwork keeps its aspect, top-left anchored, fonts scale with it. Wording and ticks: the preset in `gsro_analysis/colorbars.py` |
| **fill ramp / threshold** | `world_maps.lapse_rate_fill`: YlGnBu at `clip(rate / 8, 0, 1)`, opaque, drawn where `display_map = 1` and `snowmelt_lapse_rate_n ≥ 10`. Keep it in step with the colorbar preset |
| **hillshade / graticule** | `hillshade_decimation=4` (4 km pixels; larger = faster previews), `graticule_deg=(60, 30)` or `None`; `world_maps.HILLSHADE_STRETCH` (1–231) |
| **page, extent, scale** | `world_maps.PAGES` (`Page(width, height, map_height, extent)`; the extent's aspect must equal the map item's; the inset width and (on the sensitivity map) the font size follow the scale) |
| **output** | `out=` (`None` = do not save), `dpi=300`; `legend_png=` overrides the legend file |

**Checks.** Every call runs `world_maps.layout_report(fig, placed)` and **warns** when names overlap each other or another range's inset, when anything is off the page, or when the colorbar's ticks and label run into another picture or an inset. The report is in `placed.attrs['layout_report']`; `placed` has one row per drawn inset (page mm) so a collision can be traced to two rows.

**Where the numbers come from.** Page, extent, inset rule, fonts and picture positions reproduce the QGIS project the published polar-triplet world map was laid out in (validated side by side on 2026-09-01), then departed from it for legibility (the `STYLE_B_QGIS` style and `top_margin_mm=None` give the published look). The project, its spec and the transfer notes are archived in the private repo `recreate_global_snowmelt_runoff_onset_analysis_QGIS_figures_in_mpl`.


In [ ]:
fig, placed = world_maps.plot_lapse_rate_triplet_map(
    gmba_stats_gdf, version,
    out=figdir / 'global_lapse_rates_with_triplets_map.png',
    # knobs, defaults shown — see the cell above:
    # style=world_maps.STYLE_B, top_margin_mm=world_maps.TOP_MARGIN_MM,
    # pictures=world_maps.PICTURES['lapse_rates_with_triplets'], legend_png=None, hillshade_decimation=4, dpi=300,
)
page = placed.attrs['page']
trim = world_maps.PAGES['lapse_rates_with_triplets'].height - page.height
print(f"{len(placed)} triplet insets ({int(placed['has_png'].sum())} with a PNG); page {page.width:.0f} x {page.height:.1f} mm "
      f"(project page trimmed by {trim:.1f} mm at the top; negative = extended)")
print({k: v for k, v in placed.attrs['layout_report'].items() if v})   # collisions, if any (also raised as warnings)

Where every block landed (page mm on the rendered page; `shift_down_mm` is only non-zero with
`top_margin_mm=None`). Use it with the recipes above when moving labels.

In [ ]:
placed.set_index('MapName')[['anchor_x_mm', 'anchor_y_mm', 'inset_w_mm', 'inset_h_mm', 'block_top', 'block_bottom',
                              'callout_x1', 'callout_y1', 'shift_down_mm']].round(1).sort_index()